In [1]:
import pandas as pd
import time
import re

# あなたの代表4番打者
main_fourth_df = pd.DataFrame({
    "球団": ["DeNA", "オリックス", "ソフトバンク", "ヤクルト", "ロッテ", "中日",
           "巨人", "広島", "日本ハム", "楽天", "西武", "阪神"],
    "4番": ["牧 秀悟", "森 友哉", "山川 穂高", "村上 宗隆", "ソト", "細川 成也",
          "岡本 和真", "小園 海斗", "マルティネス", "浅村 栄斗", "佐藤 龍世", "大山 悠輔"],
    "4番出場回数": [79, 62, 143, 131, 116, 86, 143, 71, 79, 76, 32, 90]
})

TEAM_HITTER_URLS = {
    "阪神": "https://baseball-data.com/24/stats/hitter-t/",
    "DeNA": "https://baseball-data.com/24/stats/hitter-yb/",
    "巨人": "https://baseball-data.com/24/stats/hitter-g/",
    "中日": "https://baseball-data.com/24/stats/hitter-d/",
    "広島": "https://baseball-data.com/24/stats/hitter-c/",
    "ヤクルト": "https://baseball-data.com/24/stats/hitter-s/",
    "ソフトバンク": "https://baseball-data.com/24/stats/hitter-h/",
    "日本ハム": "https://baseball-data.com/24/stats/hitter-f/",
    "オリックス": "https://baseball-data.com/24/stats/hitter-bs/",
    "楽天": "https://baseball-data.com/24/stats/hitter-e/",
    "西武": "https://baseball-data.com/24/stats/hitter-l/",
    "ロッテ": "https://baseball-data.com/24/stats/hitter-m/",
}

def dedupe_repeated_label(s: str) -> str:
    """
    '選手名選手名' -> '選手名'
    'OPSOPS' -> 'OPS'
    のように、同じ文字列が2回続いた列名を1回に直す
    """
    s = str(s).replace("\n", "").replace(" ", "").replace("\u3000", "")
    n = len(s)
    if n % 2 == 0:
        half = n // 2
        if s[:half] == s[half:]:
            return s[:half]
    return s

def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = []
    for c in df.columns:
        if isinstance(c, tuple):
            c = "".join([str(x) for x in c if str(x) != "nan"])
        c = dedupe_repeated_label(str(c))
        cols.append(c)
    df = df.copy()
    df.columns = cols
    return df

def normalize_name(x):
    s = str(x)
    s = s.replace("\u3000", " ").strip()
    s = " ".join(s.split())
    return s

def load_hitter_table(url, team):
    tables = pd.read_html(url)

    for i, df in enumerate(tables):
        df = flatten_columns(df)
        print(f"{team} table {i} cols:", list(df.columns)[:20])

        cols = set(df.columns)

        # 二重列名修正後の判定
        if ("選手名" in cols) and ("OPS" in cols):
            return df

    raise ValueError(f"{team}: 打者成績表が見つかりません")

def extract_player_row(df, player_name):
    df = df.copy()

    if "選手名" not in df.columns:
        raise ValueError("選手名列が見つかりません")

    df["選手名"] = df["選手名"].astype(str).map(normalize_name)

    target = normalize_name(player_name).replace(" ", "")

    # 完全一致
    matched = df[df["選手名"].str.replace(" ", "", regex=False) == target].copy()

    # 部分一致
    if matched.empty:
        matched = df[df["選手名"].str.replace(" ", "", regex=False).str.contains(target, regex=False, na=False)].copy()

    if matched.empty:
        return None

    return matched.iloc[0]

def safe_get(row, col):
    return row[col] if col in row.index else None

results = []
not_found = []

for _, r in main_fourth_df.iterrows():
    team = r["球団"]
    player = r["4番"]
    fourth_count = r["4番出場回数"]
    url = TEAM_HITTER_URLS[team]

    print(f"\n取得中: {team} - {player}")
    try:
        df = load_hitter_table(url, team)
        row = extract_player_row(df, player)

        if row is None:
            print(f"  見つからず: {player}")
            not_found.append((team, player))
            continue

        results.append({
            "球団": team,
            "選手名": player,
            "4番出場回数": fourth_count,
            "試合": safe_get(row, "試合"),
            "打席": safe_get(row, "打席数"),
            "打数": safe_get(row, "打数"),
            "安打": safe_get(row, "安打"),
            "本塁打": safe_get(row, "本塁打"),
            "打点": safe_get(row, "打点"),
            "四球": safe_get(row, "四球"),
            "死球": safe_get(row, "死球"),
            "三振": safe_get(row, "三振"),
            "併殺打": safe_get(row, "併殺打"),
            "OBP": safe_get(row, "出塁率"),
            "SLG": safe_get(row, "長打率"),
            "OPS": safe_get(row, "OPS"),
        })

        time.sleep(1)

    except Exception as e:
        print("  ERROR:", e)
        not_found.append((team, player))

result_df = pd.DataFrame(results)

if result_df.empty:
    raise RuntimeError("1人も取得できませんでした。選手名表記を確認してください。")

result_df.to_csv("npb_2024_main_fourth_batter_stats.csv", index=False, encoding="utf-8-sig")

print("\n=== 代表4番打者の打撃成績 ===")
print(result_df)

if not_found:
    print("\n=== 見つからなかった選手 ===")
    for x in not_found:
        print(x)

print("\n保存完了: npb_2024_main_fourth_batter_stats.csv")


取得中: DeNA - 牧 秀悟
DeNA table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'OPS', 'RC27', 'XR27']

取得中: オリックス - 森 友哉
オリックス table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'OPS', 'RC27', 'XR27']

取得中: ソフトバンク - 山川 穂高
ソフトバンク table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'OPS', 'RC27', 'XR27']

取得中: ヤクルト - 村上 宗隆
ヤクルト table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'OPS', 'RC27', 'XR27']

取得中: ロッテ - ソト
ロッテ table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'OPS', 'RC27', 'XR27']

取得中: 中日 - 細川 成也
中日 table 0 cols: ['背番号', '選手名', '打率', '試合', '打席数', '打数', '安打', '本塁打', '打点', '盗塁', '四球', '死球', '三振', '犠打', '併殺打', '出塁率', '長打率', 'O